In [1]:
#!pip install pyspark==3.5.5 tables snakebite-py3

In [2]:
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"


from pyspark.sql import SparkSession

spark_session :SparkSession = SparkSession.builder \
    .master("spark://192.168.2.31:7077") \
    .appName("spark_preprocess_driver") \
    .config("spark.dynamicAllocation.enabled", True) \
    .config("spark.dynamicAllocation.shuffleTracking.enabled", True) \
    .config("spark.shuffle.service.enabled", False) \
    .config("spark.dynamicAllocation.executorIdleTimeout", "30s") \
    .config("spark.cores.max", 4) \
    .getOrCreate()
    
spark_context = spark_session.sparkContext
spark_context.setLogLevel("ERROR")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/17 17:54:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/17 17:55:01 WARN StandaloneSchedulerBackend: Dynamic allocation enabled without spark.executor.cores explicitly set, you may get more executors allocated than expected. It's recommended to set spark.executor.cores explicitly. Please check SPARK-30299 for more details.


# Get filenames in hdfs

In [3]:
from snakebite.client import Client
HDFS_HOST = "192.168.2.31"
HDFS_PORT = 9000
HDFS_BASE = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"

client = Client(HDFS_HOST, HDFS_PORT)


In [4]:
starting_directory = "/data/MillionSongSubset"

done = False

#NOTE: in case of large amounts of files, this will likely make the driver run out of memory.
#In that case, recursion has to be implemented manually. 
all_files = list(client.ls(["/data/MillionSongSubset"], recurse=True))

In [5]:
h5_files = []
for file in all_files:
    if file["file_type"] != "f":
        continue
    
    if file["path"].split(".")[-1] == "h5":
        h5_files.append(file["path"])
        

In [6]:
from hdf5_getters import get_desired
import io
import tables
import tempfile

#Change this according to needs. Check for available fields at the bottom of the hdf5_getters file
RELEVANT_FIELDS = [
    'artist_name',
    'title',
    'duration',
    'year',
    'song_id'
]

def get_relevant_metadata_of_song_file(file_path):
    binary = b''.join(list(client.cat([file_path]))[0])
    file_contents = io.BytesIO(binary)
    

    with tempfile.NamedTemporaryFile(delete=True) as temp_file:
        temp_file.write(file_contents.getvalue())
        temp_file_path = temp_file.name
        
        file = tables.open_file(temp_file_path)
        song_metadata = get_desired(file, RELEVANT_FIELDS)

    relevant_data = {}
    
    for field in RELEVANT_FIELDS:
        relevant_data[field] = str(song_metadata[field])
    
    return relevant_data


In [ ]:
#NOTE: The filtering of the metadata is done on the driver. Attempting to do it on the worker
#nodes was very complicated considering there was tons of dependencies that weren't working.
#This means that if the relevant metadata becomes larger than the drivers memory, there will be issues.
#For the current scale of the dataset, this will work. In the future, measures will have to be taken. E.g. creating multiple
#smaller dataframes and joining them or preferably making it so that the worker nodes can do this work.
relevant_metadata = []

relevant_metadata = [get_relevant_metadata_of_song_file(file) for file in h5_files] 

print(relevant_metadata[0:10])

In [8]:
print(len(h5_files))

10000


In [9]:
#df = spark_session.createDataFrame(relevant_metadata, RELEVANT_FIELDS)

In [10]:
#df.show()